# Task results gather

Loads every per-run JSON under ``results/<SLUG>/`` and assembles a
tidy comparison: macro precision/recall/F1/F2, subset accuracy, and
per-class F1, side-by-side for standard and cascading runs of the
same (species, area) experiment. Mirrors the role of ``scaling.ipynb``
for the scaling experiments.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import os

import pyrootutils

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator='pyproject.toml',
    pythonpath=True,
    dotenv=True,
)

## Parameters

In [3]:
# Slug of the (species, area) experiment to inspect. Must match the
# folder name produced by dataset_build.ipynb (task_slug output).
SLUG = 'task_s3_49_2_r50'
RESULTS_DIR = ROOT / 'results' / SLUG
print(f'gathering: {RESULTS_DIR}')
if not RESULTS_DIR.exists():
    print('  (folder missing — run train.ipynb / cascading_train.ipynb first)')

gathering: /home/nathan/Documents/multi-chirp/results/task_s3_49_2_r50


## Load every run in this experiment

Each ``*.json`` file under ``results/<SLUG>/`` becomes one row. The
``variant`` column is ``standard`` for ``train.ipynb`` outputs and
``cascading`` for ``cascading_train.ipynb`` outputs.

In [4]:
from building.geographic_task.results_grouped import gather_results, per_class_f1_table

df = gather_results(RESULTS_DIR)
print(f'{len(df)} run(s) found')
df

4 run(s) found


,file,collection,variant,model,subset_accuracy,test_loss,epochs_trained,timestamp,macro_precision,macro_recall,macro_f1,macro_f2,macro_auc_ovr,stage1_model,stage2_model,fraction_routed_to_stage2,f1__Emberiza_calandra,f1__Hippolais_polyglotta,f1__Regulus_ignicapilla,f1__non_target
0,mel_cnn.json,task_s3_49_2_r50,standard,mel_cnn,NaN,0.096699,77,2026-05-14T15:50:38.099917+00:00,0.911544,0.950969,0.929304,0.941807,0.992536,NaN,NaN,NaN,0.937803,0.902839,0.943457,0.933116
1,sincnet.json,task_s3_49_2_r50,standard,sincnet,NaN,0.109486,92,2026-05-14T10:30:11.000152+00:00,0.899019,0.940646,0.917825,0.931015,0.991263,NaN,NaN,NaN,0.929285,0.901294,0.916367,0.924354
2,cascading_sincnet__mel_cnn.json,task_s3_49_2_r50_cascading,cascading,sincnet+mel_cnn,0.875321,NaN,46,2026-05-18T11:22:47.016966+00:00,0.886820,0.885231,0.884857,0.884815,0.980010,sincnet,mel_cnn,0.773992,0.875344,0.877987,0.870588,0.915508
3,cnn1d.json,task_s3_49_2_r50,standard,cnn1d,0.746897,0.254250,88,2026-05-19T09:47:23.071521+00:00,0.783480,0.825047,0.790269,0.807989,0.952649,NaN,NaN,NaN,0.813646,0.735927,0.841817,0.769685


## Macro metrics side-by-side

In [5]:
metric_cols = [
    'macro_precision', 'macro_recall', 'macro_f1', 'macro_f2',
    'macro_auc_ovr', 'subset_accuracy',
]
available = [c for c in metric_cols if c in df.columns]
summary = df[['variant', 'model'] + available].copy()
(
    summary.style
    .background_gradient(cmap='Blues', subset=available)
    .format({c: '{:.3f}' for c in available}, na_rep='—')
    .set_caption(f'macro metrics — {SLUG}')
)

,variant,model,macro_precision,macro_recall,macro_f1,macro_f2,macro_auc_ovr,subset_accuracy
0,standard,mel_cnn,0.912,0.951,0.929,0.942,0.993,—
1,standard,sincnet,0.899,0.941,0.918,0.931,0.991,—
2,cascading,sincnet+mel_cnn,0.887,0.885,0.885,0.885,0.980,0.875
3,standard,cnn1d,0.783,0.825,0.790,0.808,0.953,0.747


## Per-class F1

Rows are runs (variant × model), columns are class names. Heat
indicates F1; pale rows highlight classes the model struggles with.

In [6]:
pc = per_class_f1_table(df)
if pc.empty:
    print('no per-class data')
else:
    display(
        pc.style
        .background_gradient(cmap='Greens', axis=None, vmin=0, vmax=1)
        .format('{:.3f}', na_rep='—')
        .set_caption(f'per-class F1 — {SLUG}')
    )